In [7]:
import getpass
from sqlalchemy import create_engine, text

url = "postgresql+psycopg://kiarash@127.0.0.1:5432/olist_db"
password = getpass.getpass("PostgreSQL password: ")
engine = create_engine(url, connect_args={"password": password})

with engine.connect() as conn:
    df = conn.execute(text("SELECT * FROM credit.v_model_input LIMIT 5")).mappings().all()
    print(df)

[{'row_id': 1, 'serious_dlqin_2yrs': 1, 'revolving_utilization_unsecured': 0.766126609, 'utilization_capped': 0.766126609, 'age': 45.0, 'num_late_30_59': 2.0, 'num_late_90': 0.0, 'num_late_60_89': 0.0, 'debt_ratio': 0.802982129, 'monthly_income': 9120.0, 'income_missing': 0, 'num_open_credit_lines': 13.0, 'num_real_estate_loans': 6.0, 'num_dependents': 2.0}, {'row_id': 2, 'serious_dlqin_2yrs': 0, 'revolving_utilization_unsecured': 0.957151019, 'utilization_capped': 0.957151019, 'age': 40.0, 'num_late_30_59': 0.0, 'num_late_90': 0.0, 'num_late_60_89': 0.0, 'debt_ratio': 0.121876201, 'monthly_income': 2600.0, 'income_missing': 0, 'num_open_credit_lines': 4.0, 'num_real_estate_loans': 0.0, 'num_dependents': 1.0}, {'row_id': 3, 'serious_dlqin_2yrs': 0, 'revolving_utilization_unsecured': 0.65818014, 'utilization_capped': 0.65818014, 'age': 38.0, 'num_late_30_59': 1.0, 'num_late_90': 1.0, 'num_late_60_89': 0.0, 'debt_ratio': 0.085113375, 'monthly_income': 3042.0, 'income_missing': 0, 'num_op

In [8]:
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

url = "postgresql+psycopg://kiarash@127.0.0.1:5432/olist_db"
password = getpass.getpass("PostgreSQL password: ")
engine = create_engine(url, connect_args={"password": password})

with engine.connect() as conn:
    frame = pd.read_sql(
        text("SELECT * FROM credit.v_model_input"),
        conn,
    )
frame.shape

(150000, 14)

In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from credit.features import build_features, model_matrix
from credit.columns import MODEL_FEATURE_COLUMNS, DERIVED_COLUMNS
from sklearn.model_selection import train_test_split

featured = build_features(frame)
base = list(MODEL_FEATURE_COLUMNS) + list(DERIVED_COLUMNS)
X, y = model_matrix(featured, base=base)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
X_train.shape, X_test.shape

((112500, 18), (37500, 18))

In [10]:
from credit.cost import select_threshold, matrix_sensitivity
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

choice = select_threshold(y_test, proba)
choice.as_dict()

{'threshold': 0.09,
 'cost_per_1000': 367.92,
 'matrix': '12:1',
 'grid_size': 201,
 'ties': 1,
 'counts': {'tp': 1787, 'fp': 5169, 'tn': 29825, 'fn': 719},
 'metrics': {'n': 37500.0,
  'positive_rate': 0.06682666666666667,
  'accuracy': 0.8429866666666667,
  'majority_class_accuracy': 0.9331733333333333,
  'precision': 0.25690051753881543,
  'recall': 0.7130885873902634,
  'f1': 0.37772141196364406,
  'specificity': 0.8522889638223695,
  'flag_rate': 0.18549333333333334},
 'undefined_metrics': []}

In [11]:
from credit.cost import CHALLENGE_MATRIX, FLATTER_MATRIX, constant_baselines, reduction_vs_baseline

sensitivity = matrix_sensitivity(y_test, proba, [CHALLENGE_MATRIX, FLATTER_MATRIX])
baselines = constant_baselines(y_test)
reduction = reduction_vs_baseline(
    baselines["flag_nobody"]["cost_per_1000"], choice.cost_per_1000
)
sensitivity, baselines, reduction

([{'matrix': '12:1',
   'ratio': 12.0,
   'threshold': 0.09,
   'cost_per_1000': 367.92,
   'cost_per_1000_at_0.5': 656.6133333333333,
   'recall': 0.7130885873902634,
   'flag_rate': 0.18549333333333334,
   'ties': 1},
  {'matrix': '4:1',
   'ratio': 4.0,
   'threshold': 0.2,
   'cost_per_1000': 181.35999999999999,
   'cost_per_1000_at_0.5': 224.39999999999998,
   'recall': 0.5163607342378292,
   'flag_rate': 0.08658666666666667,
   'ties': 1}],
 {'flag_nobody': {'name': 'flag-nobody',
   'counts': {'tp': 0, 'fp': 0, 'tn': 34994, 'fn': 2506},
   'cost_per_1000': 801.92,
   'metrics': {'n': 37500.0,
    'positive_rate': 0.06682666666666667,
    'accuracy': 0.9331733333333333,
    'majority_class_accuracy': 0.9331733333333333,
    'precision': 0.0,
    'recall': 0.0,
    'f1': 0.0,
    'specificity': 1.0,
    'flag_rate': 0.0,
    'undefined': ['precision']}},
  'flag_everybody': {'name': 'flag-everybody',
   'counts': {'tp': 2506, 'fp': 34994, 'tn': 0, 'fn': 0},
   'cost_per_1000': 933

In [ ]:
from credit.features import sentinel_spike_counts
from credit.missingness import missingness_table  # adjust import if named differently

spikes = sentinel_spike_counts(frame)
display(spikes)

# check whether income missingness carries signal
missing = frame[frame["monthly_income"].isna()]
present = frame[frame["monthly_income"].notna()]
print("default rate, income missing:", missing["serious_dlqin_2yrs"].mean())
print("default rate, income present:", present["serious_dlqin_2yrs"].mean())

ImportError: cannot import name 'missingness_table' from 'credit.missingness' (/home/rash/Code/PROJECTS/dotin/term-3/answers/stage-3-kiarash-akbari/src/credit/missingness.py)

In [13]:
import credit.missingness as m
print([name for name in dir(m) if not name.startswith("_")])

['DEFAULT_ALPHA', 'Sequence', 'TARGET', 'annotations', 'co_missingness', 'math', 'missingness_profile', 'np', 'numeric', 'pd', 'two_proportion_test']


In [21]:
import inspect
from credit.missingness import two_proportion_test
print(inspect.signature(two_proportion_test))

from credit.features import sentinel_spike_counts
from credit.missingness import missingness_profile, co_missingness, two_proportion_test

spikes = sentinel_spike_counts(frame)
display(spikes)

profile = missingness_profile(frame, columns=list(frame.columns))
display(profile)

nan_cols = [c for c in frame.columns if frame[c].isna().any()]
co = co_missingness(frame, columns=nan_cols)
display(co)
missing = frame[frame["monthly_income"].isna()]
present = frame[frame["monthly_income"].notna()]

test = two_proportion_test(
    successes_a=int(missing["serious_dlqin_2yrs"].sum()),
    total_a=len(missing),
    successes_b=int(present["serious_dlqin_2yrs"].sum()),
    total_b=len(present),
)
print(test)

(successes_a: 'int', total_a: 'int', successes_b: 'int', total_b: 'int') -> 'dict[str, float]'


,column,code,rows,share_of_file,default_rate_with_code,default_rate_without_code,base_default_rate
0,num_late_30_59,96,0,0.0,NaN,0.06684,0.06684
1,num_late_30_59,98,0,0.0,NaN,0.06684,0.06684
2,num_late_90,96,0,0.0,NaN,0.06684,0.06684
3,num_late_90,98,0,0.0,NaN,0.06684,0.06684
4,num_late_60_89,96,0,0.0,NaN,0.06684,0.06684
5,num_late_60_89,98,0,0.0,NaN,0.06684,0.06684


,column,missing_rows,missing_rate,default_rate_when_missing,default_rate_otherwise,difference,z,p_value,informative
0,row_id,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
1,serious_dlqin_2yrs,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
2,revolving_utilization_unsecured,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
3,utilization_capped,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
4,age,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
5,num_late_30_59,269,0.001793,0.546468,0.065978,0.480490,31.526410,3.775984e-218,True
6,num_late_90,269,0.001793,0.546468,0.065978,0.480490,31.526410,3.775984e-218,True
7,num_late_60_89,269,0.001793,0.546468,0.065978,0.480490,31.526410,3.775984e-218,True
8,debt_ratio,0,0.000000,NaN,NaN,NaN,NaN,NaN,False
9,monthly_income,29731,0.198207,0.056137,0.069486,-0.013349,-8.252685,1.548749e-16,True


,left,right,both_missing,left_only,right_only,neither,share_both_missing,odds_ratio
0,num_late_30_59,num_late_90,269,0,0,149731,0.001793,1.614106e+08
1,num_late_30_59,num_late_60_89,269,0,0,149731,0.001793,1.614106e+08
2,num_late_90,num_late_60_89,269,0,0,149731,0.001793,1.614106e+08
3,monthly_income,num_dependents,3924,25807,0,120269,0.026160,3.657833e+04
4,num_late_90,num_dependents,31,238,3893,145838,0.000207,4.947140e+00
5,num_late_30_59,num_dependents,31,238,3893,145838,0.000207,4.947140e+00


{'rate_a': 0.05613669234132723, 'rate_b': 0.06948590243537403, 'difference': -0.013349210094046798, 'z': -8.252685011122365, 'p_value': 1.5487486946825036e-16}


In [16]:
from credit.features import sentinel_spike_counts

spikes = sentinel_spike_counts(frame)
display(spikes)

missing = frame[frame["monthly_income"].isna()]
present = frame[frame["monthly_income"].notna()]
print("default rate, income missing:", missing["serious_dlqin_2yrs"].mean())
print("default rate, income present:", present["serious_dlqin_2yrs"].mean())

,column,code,rows,share_of_file,default_rate_with_code,default_rate_without_code,base_default_rate
0,num_late_30_59,96,0,0.0,NaN,0.06684,0.06684
1,num_late_30_59,98,0,0.0,NaN,0.06684,0.06684
2,num_late_90,96,0,0.0,NaN,0.06684,0.06684
3,num_late_90,98,0,0.0,NaN,0.06684,0.06684
4,num_late_60_89,96,0,0.0,NaN,0.06684,0.06684
5,num_late_60_89,98,0,0.0,NaN,0.06684,0.06684


default rate, income missing: 0.05613669234132723
default rate, income present: 0.06948590243537403


In [22]:
import json
from pathlib import Path

evidence_dir = Path("docs/evidence")
evidence_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "sentinel_spikes": spikes.to_dict(orient="records"),
    "missingness_profile": profile.to_dict(orient="records"),
    "co_missingness": co.to_dict(orient="records"),
    "income_missing_test": test,
}
(evidence_dir / "missingness-evidence.json").write_text(json.dumps(payload, indent=2, default=str))
print("saved:", evidence_dir / "missingness-evidence.json")

saved: docs/evidence/missingness-evidence.json


In [23]:
import json
from pathlib import Path
from credit.cost import cost_curve

evidence_dir = Path("docs/evidence")
evidence_dir.mkdir(parents=True, exist_ok=True)

curve = cost_curve(y_test, proba)

payload = {
    "chosen": {
        "matrix": choice.matrix,
        "threshold": choice.threshold,
        "cost_per_1000": choice.cost_per_1000,
        "counts": choice.counts,
        "metrics": choice.metrics,
    },
    "sensitivity": sensitivity,
    "baselines": baselines,
    "reduction_vs_baseline_pct": reduction,
    "cost_curve": {
        "threshold": curve["threshold"].tolist(),
        "cost_per_1000": curve["cost_per_1000"].tolist(),
    },
}
(evidence_dir / "credit-cost-threshold.json").write_text(json.dumps(payload, indent=2, default=str))
print("saved:", evidence_dir / "credit-cost-threshold.json")

saved: docs/evidence/credit-cost-threshold.json


In [26]:
import credit, pkgutil, importlib

for m in pkgutil.iter_modules(credit.__path__):
    mod = importlib.import_module(f"credit.{m.name}")
    funcs = [n for n in dir(mod) if not n.startswith("_") and callable(getattr(mod, n))]
    print(m.name, "->", funcs)

columns -> ['CreditColumn', 'as_records', 'csv_for', 'dataclass', 'db_for']
cost -> ['CostMatrix', 'Mapping', 'Sequence', 'ThresholdChoice', 'auc_rank', 'binary_counts', 'constant_baselines', 'cost_curve', 'cost_per_1000', 'dataclass', 'matrix_sensitivity', 'metrics', 'reduction_vs_baseline', 'rule_baseline', 'select_threshold', 'total_cost']
features -> ['Callable', 'FeatureSpec', 'Sequence', 'audit_table', 'build_features', 'dataclass', 'drop_feature', 'field', 'mask_sentinels', 'model_matrix', 'numeric', 'sentinel_spike_counts']
missingness -> ['Sequence', 'co_missingness', 'missingness_profile', 'numeric', 'two_proportion_test']
source -> ['CsvFileSource', 'DataError', 'Iterator', 'Path', 'SchemaDriftError', 'Self', 'check_page_schema', 'count_data_rows', 'file_fingerprint', 'file_sha256', 'read_header']
validate -> ['CreditColumn', 'CreditReasonCode', 'Enum', 'Iterable', 'Iterator', 'Mapping', 'ParsedRow', 'count_reasons', 'dataclass', 'parse_row', 'parse_stream']


In [28]:
from credit.features import audit_table

audit = audit_table()
display(audit)

,feature,built from,hypothesis,available before the decision,why,boundary that keeps it honest,needs evidence
0,delinquency_events_total,"num_late_30_59, num_late_90, num_late_60_89","any recorded past-due event matters, not only ...",yes,"credit bureau history, on file before the deci...",would become leakage if the window included mo...,-
1,utilization_over_limit,revolving_utilization_unsecured,reported utilisation above 100% is not a measu...,yes,bureau utilisation ratio at the report date,nothing after origination is used; the raw rat...,-
2,income_per_dependent,"monthly_income, num_dependents",income alone ignores burden; income per head o...,yes,both fields come from the application form,the constant fill for unrecorded dependents is...,-
3,overcommitted,debt_ratio,debt ratio past 1.0 means scheduled payments e...,yes,monthly obligations declared on the application,"uses only the declared ratio, never an observe...",-
4,recent_exposure_total,"num_open_credit_lines, num_real_estate_loans",a thin file - few lines of any kind - predicts...,yes,bureau counts at the report date,"counts, not ages of accounts; no post-decision...",-
5,income_not_reported,monthly_income,"if missingness carries information, the absenc...",yes,a field left blank on the form is visible at d...,"a presence indicator cannot leak an outcome, b...",missing_income_may_be_informative
